# Barranquilla MOPAC: `diagg1` con matrices densas

Con `diagg2` en denso (por defecto en QM/MM desde `bba4e09c`) un paso de tripsina–BPTI tomó 25,7 ms, y `diagg1` (los
elementos de Fock entre orbitales ocupados y virtuales y la lista de pares que rota `diagg2`) quedó como la etapa más cara
del SCF: ~5,2 ms por paso. Con `MOPAC_MOZYME_DIAGG1_DENSE=1` esos elementos salen de dos productos de matrices (cuBLAS)
sobre la matriz de Fock densa y las filas densas de los orbitales; la selección de pares es la de MOZYME (la misma
precriba con los dos primeros átomos de cada orbital, el mismo `oldlim`, el mismo orden, iteraciones pares e impares).
Solo desaparecen las cribas de términos despreciables (por debajo de ~1e-8). Si algo falla, corre el `diagg1` de siempre.

| Bloque | Qué hace | Tiempo (A100) |
|---|---|---|
| Compilar | MOPAC con OpenBLAS | ~10 min |
| B | tripsina, 400 pasos: por defecto y `diagg1` denso, dos veces cada uno, y tolerancia 0,001 | ~7 min |
| P | perfil por etapas del SCF: `diagg1` en ms por paso | ~2 min |
| N | NVE de 1 ps (tolerancia 0,001, 0,5 fs) con cada uno desde el mismo estado | ~5 min |
| B3 | chignolina entera y crambina entera: por defecto y `diagg1` denso | ~5 min |

**Comprobación:** la primera llamada es la misma geometría en todas las corridas; calor, fuerzas e iteraciones del SCF
contra el `diagg1` de siempre, y dos corridas iguales dan la escala del ruido.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a
ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '977d7e25'            # diagg1 con matrices densas (MOPAC_MOZYME_DIAGG1_DENSE=1)
RESULTS_NAME = 'qmmm_speed10_a100'
PREP_FROM = ['qmmm_speed2_a100/runs/prep', 'qmmm_dt_a100/runs/C_drift/prep', 'qmmm_guess2_a100/runs/C_drift/prep']
NVT_STEPS = 400                # pasos de 0,5 fs por variante (tripsina)
USE_DRIVE = True
QUICK = False                  # True: pocos pasos (prueba del cuaderno)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} line_profiler 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Herramientas y estados iniciales

- `/content/qmmm_prof.py`: tiempos por llamada a la fuerza QM/MM, calor, iteraciones SCF; fuerzas de la primera llamada
  (`QMMM_PROF_FORCES`).
- `/content/analyze_prof.py`: lectura de los perfiles y de `nsys`.

In [ ]:
Path('/content/qmmm_prof.py').write_text('"""Runs a QM/MM script unchanged (python qmmm_prof.py SCRIPT ARGS...) and records each call of the QM/MM force:\nstart time, duration of compute(), of MOPAC (QMSession.step) and of the long-range term, heat (kcal/mol),\nSCF iterations and MM charges seen.  QMMM_PROF_OUT: JSON file of these lists; QMMM_PROF_FORCES: .npz file of\nthe energy and forces of the first call.  QMMM_NSYS_RANGE=a,b: CUDA\nprofiler start before call a and stop after call b (nsys --capture-range=cudaProfilerApi).  QMMM_NVTX=1: NVTX\nranges mopac_step (QMSession.step) and qmmm_compute (CuPy\'s NVTX)."""\nimport json\nimport os\nimport runpy\nimport sys\nimport time\n\nscript = sys.argv[1]\nsys.argv = sys.argv[1:]\nsys.path.insert(0, os.path.dirname(os.path.abspath(script)))\nimport openmm_mopac as om  # noqa: E402\nimport mopac_qmmm_api as api  # noqa: E402\n\nT = {k: [] for k in ("t0", "compute", "mopac", "pme_far", "pme_grad", "heat", "iters", "n_mm")}\nrng = os.environ.get("QMMM_NSYS_RANGE")\nrng = tuple(int(x) for x in rng.split(",")) if rng else None\nprof = None\nif rng:\n    try:\n        import cupy\n        prof = cupy.cuda.runtime\n    except Exception as e:  # noqa: BLE001\n        print("QMMM_NSYS_RANGE: no CuPy,", e, flush=True)\n\n\ndef wrap(cls, name, key):\n    f = getattr(cls, name)\n\n    def g(self, *a, **k):\n        t = time.perf_counter()\n        try:\n            return f(self, *a, **k)\n        finally:\n            T[key].append(time.perf_counter() - t)\n    setattr(cls, name, g)\n\n\nnvtx = None\nif os.environ.get("QMMM_NVTX"):\n    try:\n        from cupy.cuda import nvtx\n    except Exception as e:  # noqa: BLE001\n        print("QMMM_NVTX: no CuPy NVTX,", e, flush=True)\n\n\ndef ranged(cls, name, label):\n    f = getattr(cls, name)\n\n    def g(self, *a, **k):\n        nvtx.RangePush(label)\n        try:\n            return f(self, *a, **k)\n        finally:\n            nvtx.RangePop()\n    setattr(cls, name, g)\n\n\nif nvtx is not None:\n    ranged(api.QMSession, "step", "mopac_step")\n    ranged(om.MopacQMMMForce, "compute", "qmmm_compute")\nwrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")\nwrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")\nwrap(api.QMSession, "step", "mopac")\n_compute = om.MopacQMMMForce.compute\n\n\ndef compute(self, state):\n    k = len(T["t0"])\n    if prof is not None and k == rng[0]:\n        prof.deviceSynchronize()\n        prof.profilerStart()\n    t = time.perf_counter()\n    T["t0"].append(t)\n    r = _compute(self, state)\n    T["compute"].append(time.perf_counter() - t)\n    T["heat"].append(self.last.get("heat"))\n    T["iters"].append(self.session.scf_iterations)\n    T["n_mm"].append(self.last.get("n_mm"))\n    if k == 0 and os.environ.get("QMMM_PROF_FORCES"):\n        import numpy as np\n        np.savez(os.environ["QMMM_PROF_FORCES"], energy=r[0], forces=np.asarray(r[1]))\n    if prof is not None and k == rng[1]:\n        prof.deviceSynchronize()\n        prof.profilerStop()\n    return r\n\n\nom.MopacQMMMForce.compute = compute\ntry:\n    runpy.run_path(script, run_name="__main__")\nfinally:\n    out = os.environ.get("QMMM_PROF_OUT")\n    if out:\n        with open(out, "w") as fh:\n            json.dump(T, fh)\n')
Path('/content/analyze_prof.py').write_text('import re\nimport sqlite3\nfrom collections import defaultdict\n\nimport numpy as np\n\nRX_SEC = re.compile(r\'\\[PROFILE\\] MOZYME_SECTION name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_STAGE = re.compile(r\'\\[PROFILE\\] MOZYME_RESIDENT_STAGE name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_TRACE = re.compile(r\'\\[MOZYME GPU SCF trace\\] diagg2 staged=(\\d+) nij=(\\d+) rotated=(\\d+) max_chain=(\\d+) \'\n                      r\'lock_spins=(\\d+) nrej=(-?\\d+) lpt=(\\d+) grid=(\\d+) span_us=([0-9.]+) wait_us=([0-9.]+) \'\n                      r\'rotate_us=([0-9.]+)\')\n\n\ndef steady(T, skip=3):\n    """per-call medians (ms) after the first `skip` calls (the first SCF starts from the Lewis structure)"""\n    c, t0, m = np.array(T[\'compute\']), np.array(T[\'t0\']), np.array(T[\'mopac\'])\n    n = len(c)\n    far = (np.array(T[\'pme_far\']) + np.array(T[\'pme_grad\'])) if len(T[\'pme_far\']) == n else np.zeros(n)\n    s = slice(skip, n)\n    gap = t0[skip + 1:] - (t0[skip:-1] + c[skip:-1])\n    it = [x for x in T[\'iters\'][skip:] if x is not None]\n    r = dict(calls=n, step_ms=(t0[-1] + c[-1] - t0[skip]) / (n - skip) * 1e3,\n             compute_ms=np.median(c[s]) * 1e3, mopac_ms=np.median(m[s]) * 1e3, far_ms=np.median(far[s]) * 1e3,\n             python_ms=np.median(c[s] - m[s] - far[s]) * 1e3, gap_ms=np.median(gap) * 1e3,\n             gap_mean_ms=np.mean(gap) * 1e3, gaps_over_50ms=int(np.sum(gap > 0.05)),\n             scf_iterations=float(np.mean(it)) if it else None, mopac_ms_per_iteration=None,\n             first_call_ms=c[0] * 1e3, first_heat=T[\'heat\'][0], first_iters=T[\'iters\'][0])\n    if it:\n        r[\'mopac_ms_per_iteration\'] = r[\'mopac_ms\'] / np.mean(it)\n    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}\n\n\ndef sections(text):\n    """MOPAC\'s cumulative section times (MOPAC_API_OUTPUT): the last value of each section"""\n    sec = {}\n    for m in RX_SEC.finditer(text):\n        sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))\n    return sec\n\n\ndef stages(text):\n    """resident SCF stages (stdout), summed over the SCF calls"""\n    tot = defaultdict(lambda: [0, 0.0])\n    for m in RX_STAGE.finditer(text):\n        tot[m.group(1)][0] += int(m.group(2))\n        tot[m.group(1)][1] += float(m.group(3))\n    return dict(tot)\n\n\ndef trace(text):\n    """diagg2 sweeps (MOPAC_MOZYME_SCF_TRACE=1): medians over the sweeps of every key=value of the trace lines"""\n    rows = []\n    for line in text.splitlines():\n        if \'[MOZYME GPU SCF trace] diagg2 \' not in line:\n            continue\n        kv = dict(re.findall(r\'(\\w+)=(-?[0-9.]+)\', line))\n        rows.append({k: float(v) for k, v in kv.items()})\n    if not rows:\n        return {}\n    keys = [k for k in rows[0] if all(k in r for r in rows)]\n    r = {\'sweeps\': len(rows)}\n    r.update({k: float(np.median([x[k] for x in rows])) for k in keys})\n    if \'span_us\' in r and \'max_chain\' in r:\n        r[\'us_per_rotation_on_critical_path\'] = float(np.median([x[\'span_us\'] / max(x[\'max_chain\'], 1)\n                                                                  for x in rows]))\n    return r\n\n\ndef trace_dense(text):\n    """dense diagg2 sweeps (MOPAC_MOZYME_DIAGG2_DENSE=1 with MOPAC_MOZYME_SCF_TRACE=1): how many were written back\n    (done) and the medians of rotations, longest chain and new atoms"""\n    rows = [{k: float(v) for k, v in re.findall(r\'(\\w+)=(-?[0-9.]+)\', line)}\n            for line in text.splitlines() if \'[MOZYME GPU SCF trace] diagg2_dense \' in line]\n    if not rows:\n        return {}\n    return dict(sweeps=len(rows), done_fraction=float(np.mean([r[\'done\'] for r in rows])),\n                rotated=float(np.median([r[\'rotated\'] for r in rows])),\n                max_chain=float(np.median([r[\'max_chain\'] for r in rows])),\n                new_atoms=float(np.median([r[\'new_atoms\'] for r in rows])))\n\n\ndef _union(iv):\n    iv = sorted(iv)\n    tot, cs, ce = 0, None, None\n    for s, e in iv:\n        if cs is None or s > ce:\n            if cs is not None:\n                tot += ce - cs\n            cs, ce = s, e\n        else:\n            ce = max(ce, e)\n    if cs is not None:\n        tot += ce - cs\n    return tot\n\n\ndef nsys_summary(sqlite_path, ncalls):\n    """GPU busy time against wall time over the captured range, launches and API calls per QM/MM call"""\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    names = dict(con.execute(\'select id, value from StringIds\'))\n    k = con.execute(\'select start, end, shortName from CUPTI_ACTIVITY_KIND_KERNEL\').fetchall()\n    mc = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMCPY\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMCPY\' in tables else []\n    ms = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMSET\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMSET\' in tables else []\n    rt = con.execute(\'select start, end, nameId from CUPTI_ACTIVITY_KIND_RUNTIME\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_RUNTIME\' in tables else []\n    t_lo = min([r[0] for r in rt] + [r[0] for r in k])\n    t_hi = max([r[1] for r in rt] + [r[1] for r in k])\n    wall = (t_hi - t_lo) * 1e-6\n    busy = _union([(s, e) for s, e, _ in k] + mc + ms) * 1e-6\n    kern = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in k:\n        kern[names.get(nid, str(nid))][0] += 1\n        kern[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    api = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in rt:\n        api[names.get(nid, str(nid))][0] += 1\n        api[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    n = max(1, ncalls)\n    return dict(calls=ncalls, wall_ms_per_call=wall / n, gpu_busy_ms_per_call=busy / n,\n                kernel_ms_per_call=sum(v[1] for v in kern.values()) / n,\n                kernels_per_call=len(k) / n, memcpy_per_call=len(mc) / n, memset_per_call=len(ms) / n,\n                api_calls_per_call=len(rt) / n,\n                top_kernels=[(name, c / n, t / n) for name, (c, t) in sorted(kern.items(), key=lambda x: -x[1][1])[:30]],\n                top_api=[(name, c / n, t / n) for name, (c, t) in sorted(api.items(), key=lambda x: -x[1][1])[:20]])\n\n\ndef _merge(iv):\n    """sorted, merged intervals"""\n    out = []\n    for s, e in sorted(iv):\n        if out and s <= out[-1][1]:\n            out[-1][1] = max(out[-1][1], e)\n        else:\n            out.append([s, e])\n    return out\n\n\ndef _overlap(merged, s, e, ends=None):\n    """length of the merged intervals inside [s, e] (ends: their end times, for a bisection)"""\n    import bisect\n    t = 0\n    k = bisect.bisect_right(ends, s) if ends is not None else 0\n    for a, b in merged[k:]:\n        if b <= s:\n            continue\n        if a >= e:\n            break\n        t += min(b, e) - max(a, s)\n    return t\n\n\ndef nvtx_timeline(sqlite_path, step_name="mopac_step", skip=2):\n    """Per NVTX range name (MOPAC\'s sections with MOPAC_NVTX=1, mopac_step from qmmm_prof.py with QMMM_NVTX=1):\n    calls, wall ms and GPU-busy ms inside, per step.  GPU idle time inside the steps, each idle interval charged\n    to the innermost section open at its midpoint.  Kernels by the section that launched them would need the\n    runtime API table; here the GPU work is matched by time only."""\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    strings = dict(con.execute(\'select id, value from StringIds\')) if \'StringIds\' in tables else {}\n    gpu = [tuple(r) for r in con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_KERNEL\')]\n    for t in (\'CUPTI_ACTIVITY_KIND_MEMCPY\', \'CUPTI_ACTIVITY_KIND_MEMSET\'):\n        if t in tables:\n            gpu += [tuple(r) for r in con.execute(f\'select start, end from {t}\')]\n    merged = _merge(gpu)\n    ends = [b for _, b in merged]\n    cols = {r[1] for r in con.execute(\'pragma table_info(NVTX_EVENTS)\')}\n    tid = \'textId\' if \'textId\' in cols else \'NULL\'\n    rows = con.execute(f\'select start, end, text, {tid} from NVTX_EVENTS where end is not null and end > start\')\n    ranges = [(s, e, t if t is not None else strings.get(i, str(i))) for s, e, t, i in rows]\n    steps = sorted((s, e) for s, e, n in ranges if n == step_name)[skip:]\n    if not steps:\n        return {\'error\': \'no %s ranges\' % step_name, \'range_names\': sorted({n for _, _, n in ranges})[:50]}\n    lo, hi = steps[0][0], steps[-1][1]\n    secs = [(s, e, n) for s, e, n in ranges if n != step_name and s >= lo and e <= hi]\n    per = defaultdict(lambda: [0, 0, 0])\n    for s, e, n in secs:\n        per[n][0] += 1\n        per[n][1] += e - s\n        per[n][2] += _overlap(merged, s, e, ends)\n    idle = defaultdict(int)\n    step_wall = step_busy = 0\n    for s, e in steps:\n        step_wall += e - s\n        step_busy += _overlap(merged, s, e, ends)\n        mine = [(ss, ee, n) for ss, ee, n in secs if ss < e and ee > s]\n        cuts = sorted({x for ss, ee, _ in mine for x in (ss, ee)})\n        gaps, cur = [], s\n        import bisect\n        for a, b in merged[bisect.bisect_right(ends, s):] + [[e, e]]:\n            if b <= s:\n                continue\n            a2 = min(max(a, s), e)\n            if a2 > cur:\n                gaps.append((cur, a2))\n            cur = max(cur, min(b, e))\n            if cur >= e:\n                break\n        for g0, g1 in gaps:             # split at section boundaries, each piece to its innermost section\n            pts = [g0] + [x for x in cuts if g0 < x < g1] + [g1]\n            for p0, p1 in zip(pts[:-1], pts[1:]):\n                mid = (p0 + p1) / 2\n                inner = [(ee - ss, n) for ss, ee, n in mine if ss <= mid <= ee]\n                idle[min(inner)[1] if inner else \'(no section)\'] += p1 - p0\n    k = len(steps)\n    return dict(steps=k, wall_ms_per_step=step_wall / k * 1e-6, gpu_busy_ms_per_step=step_busy / k * 1e-6,\n                idle_ms_per_step=(step_wall - step_busy) / k * 1e-6,\n                idle_by_section=sorted(((n, t / k * 1e-6) for n, t in idle.items()), key=lambda x: -x[1])[:40],\n                sections=sorted(((n, c / k, w / k * 1e-6, b / k * 1e-6) for n, (c, w, b) in per.items()),\n                                key=lambda x: -x[2])[:60])\n\n\ndef scf_gaps(sqlite_path, step_name="mopac_step", within="iter_resident_scf_boundary", skip=2, top=25):\n    """GPU idle intervals inside the `within` NVTX range of each step (the resident SCF loop with MOPAC_NVTX=1),\n    each with: the last NVTX mark before it on the host (stage boundaries of the resident SCF), the CUDA runtime\n    call the host was in when the GPU went idle (or host code), the GPU operations before and after it, and how\n    long after the GPU went idle the host issued the next operation.  Per step: idle time by stage and host\n    call, by kernel pair, by gap length, and the synchronizing runtime calls."""\n    import bisect\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    S = dict(con.execute(\'select id, value from StringIds\')) if \'StringIds\' in tables else {}\n    ops = [(s, e, S.get(n, str(n)), c) for s, e, n, c in\n           con.execute(\'select start, end, shortName, correlationId from CUPTI_ACTIVITY_KIND_KERNEL\')]\n    kinds = {1: \'memcpy HtoD\', 2: \'memcpy DtoH\', 8: \'memcpy DtoD\', 10: \'memcpy PtoP\'}\n    if \'CUPTI_ACTIVITY_KIND_MEMCPY\' in tables:\n        ops += [(s, e, kinds.get(k, \'memcpy %s\' % k), c) for s, e, k, c in\n                con.execute(\'select start, end, copyKind, correlationId from CUPTI_ACTIVITY_KIND_MEMCPY\')]\n    if \'CUPTI_ACTIVITY_KIND_MEMSET\' in tables:\n        ops += [(s, e, \'memset\', c) for s, e, c in\n                con.execute(\'select start, end, correlationId from CUPTI_ACTIVITY_KIND_MEMSET\')]\n    ops.sort()\n    rt = sorted((s, e, S.get(n, str(n)), c) for s, e, n, c in\n                con.execute(\'select start, end, nameId, correlationId from CUPTI_ACTIVITY_KIND_RUNTIME\'))\n    rt_start = [r[0] for r in rt]\n    launch = {c: s for s, e, n, c in rt}\n    cols = {r[1] for r in con.execute(\'pragma table_info(NVTX_EVENTS)\')}\n    tid = \'textId\' if \'textId\' in cols else \'NULL\'\n    ev = [(s, e, t if t is not None else S.get(i, str(i))) for s, e, t, i in\n          con.execute(f\'select start, end, text, {tid} from NVTX_EVENTS\')]\n    marks = sorted((s, n) for s, e, n in ev if e is None or e <= s)\n    mark_t = [m[0] for m in marks]\n    ranges = [(s, e, n) for s, e, n in ev if e is not None and e > s]\n    steps = sorted((s, e) for s, e, n in ranges if n == step_name)[skip:]\n    if not steps:\n        return {\'error\': \'no %s ranges\' % step_name}\n    win = sorted((s, e) for s, e, n in ranges if n == within and s >= steps[0][0] and e <= steps[-1][1])\n    if not win:\n        return {\'error\': \'no %s ranges inside the steps\' % within}\n\n    def short(n):\n        n = re.sub(r\'\\(.*\', \'\', n)\n        return n.replace(\'<unnamed>::\', \'\').replace(\'void \', \'\')[:60]\n    by_stage, by_pair, sizes = defaultdict(int), defaultdict(int), defaultdict(lambda: [0, 0])\n    late = []\n    k = 0\n    busy_end, prev = None, None\n    wi = 0\n    for s, e, n, c in ops:\n        if busy_end is not None and s > busy_end:\n            g0, g1 = busy_end, s\n            while wi < len(win) and win[wi][1] < g0:\n                wi += 1\n            if wi < len(win) and win[wi][0] <= g0 and g1 <= win[wi][1]:\n                d = g1 - g0\n                # split at the host\'s runtime calls and marks: each piece to the stage and the call (or host\n                # code) of its midpoint\n                r0 = max(0, bisect.bisect_right(rt_start, g0) - 1)\n                r1 = bisect.bisect_right(rt_start, g1)\n                cuts = {x for ss, ee, _, _ in rt[r0:r1] for x in (ss, ee) if g0 < x < g1}\n                cuts |= {t for t in mark_t[bisect.bisect_right(mark_t, g0):bisect.bisect_left(mark_t, g1)]}\n                pts = [g0] + sorted(cuts) + [g1]\n                for p0, p1 in zip(pts[:-1], pts[1:]):\n                    mid = (p0 + p1) / 2\n                    m = bisect.bisect_right(mark_t, mid) - 1\n                    stage = marks[m][1] if m >= 0 else \'(no mark)\'\n                    r = bisect.bisect_right(rt_start, mid) - 1\n                    api = rt[r][2] if r >= 0 and rt[r][1] >= mid else \'host code\'\n                    by_stage[(stage, re.sub(r\'_v\\d+$\', \'\', api))] += p1 - p0\n                by_pair[(short(prev), short(n))] += d\n                b = \'<5us\' if d < 5000 else \'5-20us\' if d < 20000 else \'20-100us\' if d < 100000 else \'>100us\'\n                sizes[b][0] += 1\n                sizes[b][1] += d\n                if c in launch:\n                    late.append(launch[c] - g0)\n        if busy_end is None or e > busy_end:\n            busy_end, prev = e, n\n    nsteps = len(win)\n    per = lambda t: t / nsteps * 1e-6  # noqa: E731\n    sync = defaultdict(lambda: [0, 0])\n    win_start = [w[0] for w in win]\n    for s, e, n, c in rt:\n        w = bisect.bisect_right(win_start, s) - 1\n        if w < 0 or e > win[w][1]:\n            continue\n        n2 = re.sub(r\'_v\\d+$\', \'\', n)\n        if \'Synchronize\' in n2 or n2 in (\'cudaMemcpy\', \'cudaMemset\') or \'Query\' in n2:\n            sync[n2][0] += 1\n            sync[n2][1] += e - s\n    wall = sum(b - a for a, b in win)\n    idle = sum(by_pair.values())\n    late = np.array(late) * 1e-3 if late else np.zeros(1)\n    return dict(steps=nsteps, scf_wall_ms_per_step=per(wall), scf_idle_ms_per_step=per(idle),\n                gaps_per_step={b: (v[0] / nsteps, per(v[1])) for b, v in sorted(sizes.items())},\n                host_issue_delay_us=dict(median=float(np.median(late)), p90=float(np.percentile(late, 90)),\n                                         frac_issued_after_idle=float(np.mean(late > 0))),\n                idle_by_stage_and_host_call=[(st, api, per(t)) for (st, api), t in\n                                             sorted(by_stage.items(), key=lambda x: -x[1])[:top]],\n                idle_by_kernel_pair=[(a, b, per(t)) for (a, b), t in sorted(by_pair.items(), key=lambda x: -x[1])[:top]],\n                sync_calls_per_step=sorted(((n, v[0] / nsteps, per(v[1])) for n, v in sync.items()), key=lambda x: -x[2]))\n')

sys.path.insert(0, '/content')
import analyze_prof as AP
import urllib.request
import numpy as np
PROF = Path('/content/qmmm_prof.py')
QM = SRC / 'scripts' / 'qmmm'
ND = QM / 'nve_drift.py'
DATA = SRC / 'tests' / 'data' / 'mozyme_gpu_reference'
prep = OUT / 'runs' / 'prep'
if not (prep / 'state.xml').exists():
    prep.mkdir(parents=True, exist_ok=True)
    src = next((BASE / p for p in PREP_FROM if (BASE / p / 'state.xml').exists()), None)
    if src:
        for f in ('state.xml', 'top.pdb'):
            shutil.copy2(src / f, prep / f)
        (prep / 'from.txt').write_text(str(src) + '\n')
        print('estado inicial copiado de', src)
    else:
        pdb = OUT / 'runs' / '_pdb' / '2PTC.pdb'
        pdb.parent.mkdir(parents=True, exist_ok=True)
        if not pdb.exists():
            urllib.request.urlretrieve('https://files.rcsb.org/download/2PTC.pdb', pdb)
        print('no hay estado de', PREP_FROM, '; se prepara uno nuevo (50 ps de MM)')
        run(f'{sys.executable} -u {ND} prepare {pdb} --out {prep} --padding 1.0 --variants E:57=HID '
            f'--mm-equil-ps {10 if QUICK else 50} --platform CUDA --precision mixed', log=prep / 'log.txt')
N = 40 if QUICK else NVT_STEPS
QMSEL = '--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --ron 10 --roff 12 --precision mixed'
PREV = {}                          # por defecto (diagg2 denso)
STAG = {'MOPAC_MOZYME_DIAGG1_DENSE': '1'}   # + diagg1 denso

def save_profiles(d, api_out, log_text, T):
    if T and T.get('compute'):
        (d / 'result.json').write_text(json.dumps(AP.steady(T), indent=1))
    sec = AP.sections(api_out.read_text(errors='ignore')) if api_out.exists() else {}
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    st = AP.stages(log_text)
    if st:
        with open(d / 'stages.csv', 'w') as fh:
            fh.write('stage,calls,ms_total\n')
            for k, (c, ms) in sorted(st.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        if sec:
            with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
                shutil.copyfileobj(fi, fo)
        api_out.unlink()

def nvt_run(group, name, prep_dir, qmsel, tol, envx, nvt):
    # one nve_drift.py run --nve 0 under qmmm_prof.py; returns (rc, result dict)
    d = OUT / 'runs' / group / name
    cmd = (f'{sys.executable} -u {PROF} {ND} run --prep {prep_dir} --out {d} --libmopac {LIBMOPAC} {qmsel} '
           f'--tolerance {tol} --platform CUDA --dt 0.5 --nvt {nvt} --nve 0 --seed 11')
    key = cmd + '\n' + json.dumps(envx, sort_keys=True)
    if (d / 'summary.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        print(group, name, 'ya hecho')
        return 0, json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{group}_{name}.out'
    api_out.unlink(missing_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'),
               QMMM_PROF_FORCES=str(d / 'forces0.npz'), **envx)
    t0 = time.time()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    T = json.loads((d / 'pytimes.json').read_text()) if (d / 'pytimes.json').exists() else None
    if api_out.exists():
        notes = [l for l in api_out.read_text(errors='ignore').splitlines() if '[MOZYME GPU qmmm]' in l]
        (d / 'qmmm_gpu.txt').write_text('\n'.join(notes) + '\n')
    save_profiles(d, api_out, text, T)
    r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
    print(f"{group}/{name}: rc {rc}, {time.time() - t0:.0f} s; NVT {s.get('nvt_s_per_step', float('nan')) * 1e3:.1f} "
          f"ms/step; per call (median, ms): MOPAC {r.get('mopac_ms')}, far {r.get('far_ms')}, Python {r.get('python_ms')}, "
          f"gap {r.get('gap_ms')}; SCF iterations {r.get('scf_iterations')}; first heat {r.get('first_heat')}", flush=True)
    if rc != 0:
        print('    FALLÓ; final del registro:\n' + '\n'.join(text.strip().splitlines()[-15:]))
    return rc, r
print('prep:', (prep / 'from.txt').read_text().strip() if (prep / 'from.txt').exists() else prep)

## B. Tripsina–BPTI: 400 pasos NVT de 0,5 fs por variante

In [ ]:
VARIANTS = [('prev', 0.01, PREV), ('new', 0.01, STAG), ('prev_rep2', 0.01, PREV), ('new_rep2', 0.01, STAG),
            ('prev_tol0.001', 0.001, PREV), ('new_tol0.001', 0.001, STAG)]
failed = []
for name, tol, envx in VARIANTS:
    rc, _ = nvt_run('B', name, prep, QMSEL, tol, envx, N)
    if rc:
        failed.append(name)
print('fallidas:', failed or 'ninguna')
for name, *_ in VARIANTS:
    f = OUT / 'runs' / 'B' / name / 'qmmm_gpu.txt'
    print(name, f.read_text().strip() if f.exists() else '(sin registro)')

## P. Perfil por etapas del SCF (`MOPAC_GPU_PROFILE=1`, añade sincronizaciones)

In [ ]:
for name, envx in (('prof_prev', PREV), ('prof_new', STAG)):
    nvt_run('P', name, prep, QMSEL, 0.01, dict(envx, MOPAC_GPU_PROFILE='1'), 200 if not QUICK else 20)
for name in ('prof_prev', 'prof_new'):
    f = OUT / 'runs' / 'P' / name / 'stages.csv'
    if f.exists():
        r = json.loads((OUT / 'runs' / 'P' / name / 'result.json').read_text())
        print(f'--- {name}: etapas del SCF, ms por paso')
        for st, c, ms in [l.split(',') for l in f.read_text().splitlines()[1:]][:10]:
            print(f'  {float(ms) / r["calls"]:8.3f}  {st}')

## N. NVE de 1 ps: por defecto y `diagg1` denso (tolerancia 0,001, 0,5 fs, 200 pasos de NVT antes)

`slope_total`: pendiente de la energía total (kJ/mol/ps); `slope_qm_nonconservative`: la parte de la fuerza QM/MM que no
conserva (energía de la fuerza más su trabajo), con sus errores estándar.

In [ ]:
for name, envx in (('nve_prev', PREV), ('nve_new', STAG)):
    d = OUT / 'runs' / 'N' / name
    if not (d / 'summary.json').exists():
        shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
        env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1', **envx)
        cmd = (f'{sys.executable} -u {ND} run --prep {prep} --out {d} --libmopac {LIBMOPAC} {QMSEL} --tolerance 0.001 '
               f'--platform CUDA --dt 0.5 --nvt {20 if QUICK else 200} --nve {100 if QUICK else 2000} --seed 11')
        (d / 'cmd.txt').write_text(cmd)
        with open(d / 'log.txt', 'w') as fh:
            rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
        print(name, 'rc', rc)
    if (d / 'summary.json').exists():
        sm = json.loads((d / 'summary.json').read_text())
        print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in sm.items()
                     if k.startswith(('slope', 'nve_s', 'nvt_s', 'drift', 'scf'))})

## B3. Chignolina entera y crambina entera: por defecto y `diagg1` denso

In [ ]:
BIG = [('chignolin', DATA / '1UAO_opt.pdb', 2.5, ','.join(map(str, range(1, 11))), 20 if QUICK else 200),
       ('crambin', DATA / '1CRN_opt.pdb', 3.0, ','.join(map(str, range(1, 47))), 10 if QUICK else 40)]
for mol, pdb, pad, res, nvt in BIG:
    pdir = OUT / 'runs' / 'B3' / f'prep_{mol}'
    old = BASE / 'qmmm_speed2_a100' / 'runs' / 'B3' / f'prep_{mol}'
    if not (pdir / 'state.xml').exists():
        pdir.mkdir(parents=True, exist_ok=True)
        if (old / 'state.xml').exists():
            for f in ('state.xml', 'top.pdb'):
                shutil.copy2(old / f, pdir / f)
            (pdir / 'from.txt').write_text(str(old) + '\n')
        else:
            run(f'{sys.executable} -u {ND} prepare {pdb} --out {pdir} --padding {pad} --mm-equil-ps {2 if QUICK else 10} '
                f'--platform CUDA --precision mixed', check=False, log=pdir / 'log.txt')
    if not (pdir / 'state.xml').exists():
        print(mol, ': no se pudo preparar'); continue
    sel = f'--qm-residues {res} --ron 6 --roff 8 --precision mixed'
    for name, envx in (('prev', PREV), ('new', STAG)):
        nvt_run('B3', f'{mol}_{name}', pdir, sel, 0.01, envx, nvt)
        f = OUT / 'runs' / 'B3' / f'{mol}_{name}' / 'qmmm_gpu.txt'
        print(mol, name, f.read_text().strip() if f.exists() else '(sin registro)')

## D. Resultados

`speed10_runs.csv`: por corrida, tiempo por paso, medianas por llamada (MOPAC, largo alcance, Python, hueco),
iteraciones SCF; y, contra la variante «como antes» de la misma molécula y tolerancia (`prev`): diferencia del calor y
mayor diferencia de fuerza de la primera llamada (misma geometría), en kcal/mol y kJ/(mol nm).

In [ ]:
import pandas as pd
pd.set_option('display.width', 250)
rows = []
for group in ('B', 'P', 'B3'):
    gdir = OUT / 'runs' / group
    if not gdir.exists():
        continue
    for d in sorted(x for x in gdir.iterdir() if x.is_dir() and not x.name.startswith('prep')):
        r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
        s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
        mol = 'trypsin' if group in ('B', 'P') else d.name.split('_')[0]
        variant = d.name if group in ('B', 'P') else d.name.split('_', 1)[1]
        r.update(group=group, molecule=mol, variant=variant, nvt_ms_per_step=s.get('nvt_s_per_step', np.nan) * 1e3,
                 ok=(d / 'summary.json').exists())
        if (d / 'forces0.npz').exists():
            ref_name = 'prev_tol0.001' if variant.endswith('tol0.001') else 'prev'
            ref = OUT / 'runs' / ('B' if group == 'P' else group) / (ref_name if group in ('B', 'P') else f'{mol}_prev') / 'forces0.npz'
            if ref.exists():
                a, b = np.load(d / 'forces0.npz'), np.load(ref)
                r['first_heat_diff_vs_prev'] = float(a['energy'] - b['energy']) / 4.184
                r['first_force_maxdiff_vs_prev'] = float(np.abs(a['forces'] - b['forces']).max())
                r['first_force_max'] = float(np.abs(b['forces']).max())
        rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'speed10_runs.csv', index=False)
display(df[[c for c in ['molecule', 'variant', 'ok', 'step_ms', 'mopac_ms', 'far_ms', 'python_ms', 'gap_ms',
                        'scf_iterations', 'first_heat', 'first_heat_diff_vs_prev', 'first_force_maxdiff_vs_prev',
                        'first_force_max'] if c in df]])
print('calor de la primera llamada, tripsina, contra los perfiles anteriores (-297.0127265):',
      {r['variant']: round(r['first_heat'] + 297.0127265, 7) for r in rows
       if r.get('molecule') == 'trypsin' and r.get('first_heat') and 'tol0.001' not in r['variant']})

## E. Descargar

In [ ]:
tmp = Path('/content/_qmmm_speed10') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('speed10_runs.csv', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', '*.csv', '*.txt', 'forces0.npz', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts or f.name == 'state.xml':
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)